# Lektion 05 - Agentisches RAG


## Einrichtung

Dieses Notebook demonstriert das Agentic RAG (Retrieval-Augmented Generation) Muster unter Verwendung des Microsoft Agent Frameworks.

**Voraussetzungen:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — Ihr Azure AI Search Service-Endpunkt
- `AZURE_SEARCH_API_KEY` — Ihr Azure AI Search API-Schlüssel
- Azure OpenAI Bereitstellung, konfiguriert über Umgebungsvariablen
- Azure CLI authentifiziert (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Was ist Agentic RAG?

Traditionelles RAG folgt einer festen Pipeline: Dokumente abrufen, dann eine Antwort generieren. **Agentic RAG** geht einen Schritt weiter, indem es dem Agenten die Autonomie gibt, zu entscheiden, **wann** und **wie** Informationen abgerufen werden.

Mit Agentic RAG kann der Agent:
- **Entscheiden**, ob eine Informationsabfrage vor der Beantwortung einer Frage notwendig ist
- **Auswählen**, welche Datenquelle oder welches Tool abgefragt werden soll
- **Bewerten**, ob die abgerufenen Ergebnisse ausreichen, und bei unzureichendem Ergebnis weitere Abrufe durchführen
- **Kombinieren** von Informationen aus mehreren Abrufschritten zu einer kohärenten Antwort

Dies macht den Agenten flexibler und genauer im Vergleich zu einer statischen Retrieve-then-Generate-Pipeline.


## Erstellen eines Suchwerkzeugs

In Agentic RAG werden externe Datenquellen als **Werkzeuge** verpackt, die der Agent bei Bedarf aufrufen kann. Dadurch kann der Agent die Abfrage einfach als eine weitere Aktion behandeln, die er ausführen kann, und nicht als einen verpflichtenden Schritt.

Unten definieren wir eine Wissensdatenbank für Reisen und stellen sie als Werkzeug bereit, das der Agent aufrufen kann, um Informationen zu Reisezielen nachzuschlagen.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Den RAG-Agenten erstellen

Jetzt erstellen wir einen Agenten, der angewiesen ist, **immer Informationen abzurufen, bevor er antwortet**. Der Agent verwendet das Tool `search_travel_knowledge`, um seine Antworten in der Wissensdatenbank zu verankern, anstatt sich auf seine eigenen Trainingsdaten zu verlassen.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterative Retrieval — Das Maker-Checker-Muster

Ein wesentlicher Vorteil von Agentic RAG ist die **iterative Retrieval**. Der Agent kann mehrere Suchdurchläufe durchführen, um seine anfänglichen Ergebnisse zu überprüfen, zu verfeinern oder zu erweitern — ähnlich einem „Maker-Checker“-Arbeitsablauf:

1. **Maker-Schritt**: Der Agent ruft erste Informationen ab und erstellt einen Entwurf der Antwort.
2. **Checker-Schritt**: Der Agent führt weitere Abrufe durch, um Details zu überprüfen oder Lücken zu schließen.

Im Folgenden wird dem Agenten eine Frage gestellt, die den Vergleich mehrerer Reiseziele erfordert und ihn dazu veranlasst, mehrmals zu suchen.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Zusammenfassung

In dieser Lektion haben Sie gelernt, wie man ein **Agentic RAG**-System mithilfe des Microsoft Agent Frameworks erstellt:

- **Agentic RAG** ermöglicht es Agenten, autonom zu entscheiden, wann Informationen abgerufen werden sollen, wodurch der Abruf dynamisch statt statisch wird.
- **Tools als Datenquellen**: Externe Wissensdatenbanken (wie Azure AI Search) werden als Tools eingebunden, die der Agent aufrufen kann.
- **Iterativer Abruf**: Das Maker-Checker-Muster ermöglicht dem Agenten, mehrere Abrufrunden durchzuführen — suchen, prüfen und verfeinern — bevor eine endgültige Antwort erzeugt wird.

In der Produktion würden Sie die im Speicher liegende `TRAVEL_KNOWLEDGE_BASE` durch einen echten Azure AI Search Index ersetzen, um eine großflächige Reise-Dokumentenabfrage zu ermöglichen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:  
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, kann es bei automatisierten Übersetzungen zu Fehlern oder Ungenauigkeiten kommen. Das Originaldokument in seiner ursprünglichen Sprache gilt als maßgebliche Quelle. Für wichtige Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
